In [3]:
# @title 🚑 RELAY — Bridging the Gap Before Arrival (Login Screen + Live Voice + 100% Bilingual)
!pip install -q google-genai pydantic requests

import os, re, csv, json, time, uuid, base64, socket, threading, getpass
from datetime import datetime
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse
from typing import Optional, List, Dict, Any
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
import requests

def get_secret_or_env(key: str, prompt_if_missing: bool = False) -> str:
    val = os.environ.get(key, "").strip()
    if not val:
        try:
            from google.colab import userdata
            val = (userdata.get(key) or "").strip()
        except Exception:
            val = ""
    if not val and prompt_if_missing:
        val = getpass.getpass(f"🔑 Enter your {key}: ").strip()
    if val:
        os.environ[key] = val
    return val

GEMINI_API_KEY = get_secret_or_env("GEMINI_API_KEY", prompt_if_missing=True)
TELEGRAM_BOT_TOKEN = get_secret_or_env("TELEGRAM_BOT_TOKEN", prompt_if_missing=False)
TELEGRAM_CHAT_ID = get_secret_or_env("TELEGRAM_CHAT_ID", prompt_if_missing=False)

client = genai.Client(api_key=GEMINI_API_KEY)
MODELS_PRIORITY = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-flash-lite-latest"]
LOG_FILE = "relay_handover_audit_log.csv"
CASES_STORE: List[Dict[str, Any]] = []
STORE_LOCK = threading.Lock()

VALID_PARAMEDICS = {
    "EMS001": {"pin": "2580", "name": "Paramedic 1 (EMS001)"},
    "EMS002": {"pin": "1470", "name": "Paramedic 2 (EMS002)"},
}

def redact_phi_for_llm(text: str) -> str:
    if not text:
        return ""
    cleaned = re.sub(r"\b\d{14}\b", "[NATIONAL_ID_REDACTED]", text)
    cleaned = re.sub(r"(?:\+?20|0)?1[0125]\d{8}\b", "[PHONE_REDACTED]", cleaned)
    cleaned = re.sub(
        r"(?i)(patient\s+name\s+is|name\s+is|المريض|اسمه|اسمها)\s+([A-Za-z\u0600-\u06FF]+(?:\s+[A-Za-z\u0600-\u06FF]+){0,2})",
        r"\1 [PATIENT_NAME_REDACTED]",
        cleaned,
    )
    return cleaned

def mask_national_id(nid: str) -> str:
    digits = re.sub(r"\D", "", str(nid or ""))
    if len(digits) >= 4:
        return "*" * max(10, len(digits) - 4) + digits[-4:]
    return "**************" if digits else "N/A"

def generate_patient_id() -> str:
    date_part = datetime.now().strftime("%Y%m%d")
    with STORE_LOCK:
        seq = len(CASES_STORE) + 1
    suffix = uuid.uuid4().hex[:3].upper()
    return f"PT-{date_part}-{seq:03d}-{suffix}"

class ExtractedCaseFields(BaseModel):
    patient_name: Optional[str] = Field(default=None)
    age: Optional[int] = Field(default=None)
    gender: Optional[str] = Field(default=None)
    national_id: Optional[str] = Field(default=None)
    sbp: Optional[int] = Field(default=None)
    dbp: Optional[int] = Field(default=None)
    hr: Optional[int] = Field(default=None)
    spo2: Optional[int] = Field(default=None)
    rr: Optional[int] = Field(default=None)
    temp: Optional[float] = Field(default=None)
    avpu: Optional[str] = Field(default=None)
    gcs: Optional[int] = Field(default=None)
    medications: Optional[str] = Field(default=None)
    eta: Optional[int] = Field(default=None)

class ClinicalDraftResponse(BaseModel):
    clinical_summary: str
    suggested_department: str
    immediate_actions: List[str]
    staff_to_alert: List[str]
    equipment: List[str]
    labs_imaging: List[str]
    medications_prep: List[str]
    precautions: List[str]

def assess_vitals(data: Dict[str, Any], lang: str = "en") -> Dict[str, Any]:
    is_en = (lang == "en")
    sbp = data.get("sbp")
    dbp = data.get("dbp")
    hr = data.get("hr")
    spo2 = data.get("spo2")
    rr = data.get("rr")
    temp = data.get("temp")
    avpu = (data.get("avpu") or "").upper()
    gcs = data.get("gcs")
    age = data.get("age")

    map_val = round(dbp + (sbp - dbp) / 3.0) if (sbp and dbp and sbp > 0 and dbp > 0) else None
    si_val = round(hr / float(sbp), 2) if (hr and sbp and sbp > 0) else None

    alerts = []
    min_score = 2
    tier = "GREEN"

    def escalate(new_tier: str, score_floor: int):
        nonlocal tier, min_score
        order = {"GREEN": 1, "ORANGE": 2, "RED": 3}
        if order[new_tier] > order[tier]:
            tier = new_tier
        if score_floor > min_score:
            min_score = score_floor

    if spo2 is not None:
        if spo2 < 90:
            escalate("RED", 9)
            alerts.append(f"SpO2 {spo2}% — Severe hypoxia (<90%)" if is_en else f"نقص أكسجة حاد SpO2 {spo2}% (<90%)")
        elif spo2 <= 94:
            escalate("ORANGE", 6)
            alerts.append(f"SpO2 {spo2}% — Hypoxia range (90–94%)" if is_en else f"نقص أكسجة متوسط SpO2 {spo2}% (90–94%)")

    if sbp is not None:
        if sbp < 90:
            escalate("RED", 9)
            alerts.append(f"SBP {sbp} mmHg — Hypotension / Shock threshold" if is_en else f"الضغط الانقباضي {sbp} — هبوط ضغط / صدمة")
        elif sbp <= 100 or sbp >= 180:
            escalate("ORANGE", 6)
            alerts.append(f"SBP {sbp} mmHg — Borderline hemodynamic instability" if is_en else f"الضغط الانقباضي {sbp} — عدم استقرار ديناميكي")

    if map_val is not None and map_val < 65:
        escalate("RED", 9)
        alerts.append(f"Estimated MAP {map_val} mmHg — Low organ perfusion (<65)" if is_en else f"متوسط الضغط الشرياني MAP={map_val} mmHg — انخفاض التروية")

    if hr is not None:
        if hr > 130 or hr < 45:
            escalate("RED", 9)
            alerts.append(f"Heart Rate {hr} bpm — Critical tachycardia/arrhythmia" if is_en else f"النبض {hr}/د — تسارع/اضطراب نظم حرج")
        elif hr > 110 or hr < 55:
            escalate("ORANGE", 6)
            alerts.append(f"Heart Rate {hr} bpm — Elevated heart rate (>110)" if is_en else f"النبض {hr}/د — تسارع نبض فوق الحد")

    if si_val is not None:
        if si_val > 1.2:
            escalate("RED", 9)
            alerts.append(f"Shock Index SI={si_val} (HR/SBP) — Exceeds 1.2 shock threshold" if is_en else f"مؤشر الصدمة SI={si_val} — تجاوز عتبة 1.2 (تنبيه صدمة)")
        elif si_val >= 1.0:
            escalate("ORANGE", 6)
            alerts.append(f"Shock Index SI={si_val} — Early shock warning" if is_en else f"مؤشر الصدمة SI={si_val} — مؤشر صدمة مبكر")

    if rr is not None:
        if rr > 28 or rr < 10:
            escalate("RED", 9)
            alerts.append(f"Respiratory Rate {rr}/min — Severe respiratory distress" if is_en else f"معدل التنفس {rr}/د — ضائقة تنفسية حادة")
        elif rr > 22:
            escalate("ORANGE", 6)
            alerts.append(f"Respiratory Rate {rr}/min — Tachypnea (>22)" if is_en else f"معدل التنفس {rr}/د — تسرع تنفس")

    if avpu in ("P", "U"):
        escalate("RED", 9)
        alerts.append(f"AVPU = {avpu} — Severe altered mental status" if is_en else f"مستوى الوعي AVPU={avpu} — تدهور شديد في الوعي")
    elif avpu == "V":
        escalate("ORANGE", 7)
        alerts.append("AVPU = V — Responds only to verbal stimuli" if is_en else "مستوى الوعي AVPU=V — يستجيب للصوت فقط")

    if gcs is not None:
        if gcs <= 8:
            escalate("RED", 10)
            alerts.append(f"GCS {gcs}/15 — Coma / Airway protection required" if is_en else f"مقياس غلاسكو GCS={gcs}/15 — غيبوبة")
        elif gcs <= 13:
            escalate("ORANGE", 7)
            alerts.append(f"GCS {gcs}/15 — Moderate neurological impairment" if is_en else f"مقياس غلاسكو GCS={gcs}/15 — اضطراب متوسط في الوعي")

    if temp is not None and (temp >= 39.5 or temp <= 35.0):
        escalate("ORANGE", 6)
        alerts.append(f"Temperature {temp}°C — Critical thermal abnormality" if is_en else f"الحرارة {temp}°C — اضطراب حراري حرج")

    if age is not None and age < 14 and tier != "GREEN":
        alerts.append("Pediatric patient (<14y) — Pediatric thresholds applied" if is_en else "مريض طفل (<14 سنة) — تطبيق حدود الأطفال")

    if not alerts:
        alerts.append("All recorded vital signs within stable physiological limits." if is_en else "جميع العلامات الحيوية المسجلة ضمن الحدود المستقرة.")

    return {
        "tier": tier,
        "score": 10 if tier == "RED" else min_score,
        "map": map_val,
        "shock_index": si_val,
        "alerts": alerts,
    }

def call_gemini_structured(contents: Any, schema_cls: Any, system_instruction: str):
    last_err = None
    for model_name in MODELS_PRIORITY:
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    response_mime_type="application/json",
                    response_schema=schema_cls,
                    temperature=0.1,
                ),
            )
            return schema_cls.model_validate_json(resp.text), model_name
        except Exception as e:
            last_err = str(e)
            continue
    raise RuntimeError(last_err or "All Gemini models failed")

def api_extract_fields(text: str, lang: str = "en", mode: str = "deep") -> Dict[str, Any]:
    sys_prompt = (
        "You are a precision medical emergency data extractor for RELAY EMS Triage. "
        "Extract clinical fields accurately from Egyptian Arabic or English paramedic speech/notes. "
        "Rules: 1) Normalize digits to standard integers/floats. "
        "2) Map gender strictly to 'Male' or 'Female' if stated, else null. "
        "3) Extract national_id as 14 digits without spaces if present. "
        "4) For medications, preserve both named drugs (e.g., Aspirin, Metformin, Plavix) AND unspecified drug classes "
        "(e.g., 'unknown blood thinners'). "
        "5) Return null for any field not explicitly mentioned."
    )
    parsed, model_used = call_gemini_structured(
        contents=f"MODE={mode}\nUI_LANG={lang}\nTRANSCRIPT:\n{text}",
        schema_cls=ExtractedCaseFields,
        system_instruction=sys_prompt,
    )
    return {"ok": True, "fields": parsed.model_dump(), "model": model_used}

def api_transcribe_audio(audio_b64: str, mime_type: str, lang: str = "en") -> Dict[str, Any]:
    raw_bytes = base64.b64decode(audio_b64)
    prompt = (
        "Transcribe this paramedic emergency voice handover verbatim. "
        f"Expected primary language: {'English' if lang == 'en' else 'Egyptian Arabic'}. "
        "Output ONLY the exact spoken transcript."
    )
    last_err = None
    for model_name in MODELS_PRIORITY:
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=[types.Part.from_bytes(data=raw_bytes, mime_type=mime_type or "audio/webm"), prompt],
                config=types.GenerateContentConfig(temperature=0.0),
            )
            return {"ok": True, "transcript": (resp.text or "").strip(), "model": model_name}
        except Exception as e:
            last_err = str(e)
            continue
    return {"ok": False, "error": last_err or "Audio transcription failed"}

def api_generate_draft(payload: Dict[str, Any]) -> Dict[str, Any]:
    lang = payload.get("lang", "en")
    is_en = (lang == "en")
    vitals_eval = assess_vitals(payload, lang=lang)
    patient_id = generate_patient_id()
    safe_complaint = redact_phi_for_llm(payload.get("complaint", ""))

    out_lang_rule = (
        "Write EVERY single field 100% in clear clinical ENGLISH. Do NOT include any Arabic characters."
        if is_en else
        "Write EVERY single field 100% in clear clinical ARABIC."
    )
    sys_prompt = f"You are RELAY's clinical handover assistant generating a preliminary ER readiness draft. {out_lang_rule} Keep lists concise (2-5 approved clinical items per category)."

    clinical_context = {
        "age": payload.get("age"),
        "gender": payload.get("gender"),
        "vitals": {
            "SBP": payload.get("sbp"), "DBP": payload.get("dbp"), "MAP": vitals_eval["map"],
            "HR": payload.get("hr"), "Shock_Index": vitals_eval["shock_index"],
            "SpO2": payload.get("spo2"), "RR": payload.get("rr"), "Temp_C": payload.get("temp"),
            "AVPU": payload.get("avpu"), "GCS": payload.get("gcs"),
        },
        "deterministic_tier": vitals_eval["tier"],
        "deterministic_alerts": vitals_eval["alerts"],
        "medications": payload.get("medications"),
        "eta_minutes": payload.get("eta"),
        "redacted_complaint": safe_complaint,
    }

    try:
        draft_obj, model_used = call_gemini_structured(
            contents=json.dumps(clinical_context, ensure_ascii=False),
            schema_cls=ClinicalDraftResponse,
            system_instruction=sys_prompt,
        )
        draft_dict = draft_obj.model_dump()
    except Exception:
        model_used = "Rules-Only Fallback Engine"
        draft_dict = {
            "clinical_summary": (
                f"{payload.get('age') or '?'}y {payload.get('gender') or ''} patient, Tier {vitals_eval['tier']}, BP {payload.get('sbp')}/{payload.get('dbp')}, HR {payload.get('hr')}, SpO2 {payload.get('spo2')}%, ETA {payload.get('eta')}m."
                if is_en else
                f"مريض عمر {payload.get('age') or '?'} سنة، الفئة {vitals_eval['tier']}، الضغط {payload.get('sbp')}/{payload.get('dbp')}، النبض {payload.get('hr')}."
            ),
            "suggested_department": "Emergency - Resuscitation Room" if is_en else "الطوارئ - غرفة الإنعاش والحالات الحرجة",
            "immediate_actions": ["Secure airway & O2", "Establish IV access", "Continuous ECG monitor"] if is_en else ["تأمين مجرى الهواء", "تركيب خط وريدي", "توصيل مونيتور"],
            "staff_to_alert": ["Emergency Physician", "Cardiologist", "ER Nurse"] if is_en else ["طبيب طوارئ", "طبيب قلب", "تمريض طوارئ"],
            "equipment": ["Cardiac Monitor", "Defibrillator / Crash Cart"] if is_en else ["جهاز مونيتور", "عربة إنعاش"],
            "labs_imaging": ["12-Lead ECG", "Troponin", "CBC & ABG"] if is_en else ["تخطيط قلب فوري", "تروبونين", "تحاليل دم وغازات"],
            "medications_prep": ["Oxygen", "IV Crystalloids"] if is_en else ["أكسجين", "محلول وريدي"],
            "precautions": vitals_eval["alerts"][:2],
        }

    return {"ok": True, "patient_id": patient_id, "model": model_used, "vitals_eval": vitals_eval, "draft": draft_dict}

def send_telegram_alert(record: Dict[str, Any]) -> bool:
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return False
    if record.get("tier") not in ("RED", "ORANGE"):
        return False
    tier_icon = {"RED": "🔴", "ORANGE": "🟠"}.get(record["tier"], "⚪")
    msg = (
        f"{tier_icon} *RELAY PRE-ARRIVAL ALERT ({record['tier']})*\n"
        f"🆔 *Patient ID:* `{record['patient_id']}`\n"
        f"👤 *Patient:* {record['patient_name']} | {record['age']}y | {record['gender']}\n"
        f"⏱️ *ETA:* {record['eta']} min | 🏥 *Dept:* {record['department']}\n"
        f"🩺 *Vitals:* BP {record['sbp']}/{record['dbp']} (MAP {record['map']}) | HR {record['hr']} | SpO2 {record['spo2']}%\n"
        f"💊 *Meds:* {record['medications'] or 'None'}\n"
        f"📋 *Summary:* {record['summary']}\n"
        f"✅ *Verified By:* {record['paramedic']}"
    )
    try:
        r = requests.post(
            f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage",
            json={"chat_id": TELEGRAM_CHAT_ID, "text": msg, "parse_mode": "Markdown"},
            timeout=6,
        )
        return r.status_code == 200
    except Exception:
        return False

def api_confirm_case(record: Dict[str, Any]) -> Dict[str, Any]:
    record["masked_nid"] = mask_national_id(record.get("national_id", ""))
    record["confirmed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    record.pop("national_id", None)
    with STORE_LOCK:
        CASES_STORE.insert(0, record)

    file_exists = os.path.isfile(LOG_FILE)
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["timestamp", "patient_id", "tier", "score", "patient_name", "age", "gender", "masked_nid", "sbp", "dbp", "map", "hr", "spo2", "eta", "department", "manually_edited", "paramedic", "model"])
        writer.writerow([
            record["confirmed_at"], record["patient_id"], record["tier"], record["score"],
            record["patient_name"], record["age"], record["gender"], record["masked_nid"],
            record["sbp"], record["dbp"], record["map"], record["hr"], record["spo2"],
            record["eta"], record["department"], record.get("manually_edited", False),
            record["paramedic"], record["model"]
        ])
    tg_sent = send_telegram_alert(record)
    return {"ok": True, "telegram_sent": tg_sent, "record": record}

HTML_PAGE = r"""<!DOCTYPE html>
<html lang="en" dir="ltr">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>RELAY — Bridging the Gap Before Arrival</title>
<style>
  :root {
    --navy: #0f2942; --blue: #0284c7; --bg: #f1f5f9; --card: #ffffff;
    --border: #cbd5e1; --text: #0f172a; --muted: #475569;
    --red: #dc2626; --orange: #ea580c; --green: #16a34a;
  }
  * { box-sizing: border-box; font-family: 'Segoe UI', Tahoma, Arial, sans-serif; }
  body { margin: 0; background: var(--bg); color: var(--text); }
  .topbar {
    background: linear-gradient(135deg, #0b1f3a, #133b66); color: #fff;
    padding: 14px 22px; display: flex; justify-content: space-between; align-items: center;
    box-shadow: 0 3px 10px rgba(0,0,0,0.15);
  }
  .brand h1 { margin: 0; font-size: 22px; letter-spacing: 0.5px; }
  .brand small { color: #93c5fd; font-size: 12.5px; }
  .nav-actions { display: flex; gap: 10px; align-items: center; }
  .btn {
    border: none; border-radius: 8px; padding: 9px 15px; font-weight: 600;
    cursor: pointer; font-size: 13px; transition: 0.15s;
  }
  .btn:disabled { opacity: 0.6; cursor: not-allowed; }
  .btn-lang { background: #1e293b; color: #fff; border: 1px solid #475569; }
  .btn-er { background: #0284c7; color: #fff; }
  .btn-primary { background: #0284c7; color: #fff; width: 100%; font-size: 15px; padding: 12px; }
  .btn-dark { background: #0f172a; color: #fff; }
  .btn-danger { background: #dc2626; color: #fff; }
  .btn-outline { background: #fff; border: 1px solid var(--border); color: var(--text); }
  .container { max-width: 1120px; margin: 18px auto; padding: 0 16px 40px; }
  .login-wrap { max-width: 450px; margin: 65px auto; padding: 0 16px; }
  .login-card {
    background: #fff; border: 1px solid var(--border); border-radius: 16px;
    padding: 28px 26px; box-shadow: 0 10px 28px rgba(15,23,42,0.08);
  }
  .card {
    background: var(--card); border: 1px solid var(--border); border-radius: 12px;
    padding: 16px 18px; margin-bottom: 14px; box-shadow: 0 1px 4px rgba(15,23,42,0.04);
  }
  .card h3 { margin: 0 0 12px; font-size: 15px; color: var(--navy); border-bottom: 1px solid #e2e8f0; padding-bottom: 8px; }
  .disclaimer {
    background: #fefce8; border: 1px solid #fde047; color: #854d0e;
    padding: 10px 14px; border-radius: 10px; font-size: 13px; margin-bottom: 12px;
  }
  .warn-transcript {
    background: #fffbeb; border: 1px solid #fcd34d; color: #92400e;
    padding: 7px 11px; border-radius: 7px; font-size: 12px; margin-top: 6px; display: none;
  }
  .status-strip {
    background: #ecfdf5; border: 1px solid #6ee7b7; color: #065f46;
    padding: 8px 14px; border-radius: 8px; font-size: 12.5px; margin-bottom: 14px;
    display: flex; justify-content: space-between; flex-wrap: wrap; gap: 8px;
  }
  .voice-box {
    background: #e0f2fe; border: 1px solid #7dd3fc; border-radius: 12px;
    padding: 14px 16px; margin-bottom: 14px;
  }
  .interim-box {
    margin-top: 8px; background: #fff; border: 1px dashed #0284c7; border-radius: 8px;
    padding: 8px 12px; font-size: 13px; color: #0369a1; min-height: 36px;
  }
  .grid-4 { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; }
  .grid-3 { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }
  .grid-2 { display: grid; grid-template-columns: repeat(2, 1fr); gap: 12px; }
  @media (max-width: 820px) { .grid-4, .grid-3, .grid-2 { grid-template-columns: 1fr; } }
  label { display: block; font-size: 12.5px; font-weight: 600; margin-bottom: 5px; color: #334155; }
  input, select, textarea {
    width: 100%; padding: 9px 11px; border: 1px solid var(--border); border-radius: 8px;
    font-size: 13.5px; background: #fff; color: var(--text);
  }
  input.edited, select.edited, textarea.edited { border-color: #d97706; background: #fffbeb; }
  .edit-tag { font-size: 11px; color: #b45309; font-weight: 600; display: none; margin-top: 2px; }
  .map-box {
    background: #f8fafc; border: 1px solid #cbd5e1; border-radius: 10px;
    padding: 10px; text-align: center; display: flex; flex-direction: column; justify-content: center;
  }
  .map-val { font-size: 18px; font-weight: 800; color: var(--navy); }
  .chip {
    background: #e0f2fe; color: #0369a1; border: 1px solid #bae6fd;
    padding: 5px 10px; border-radius: 6px; font-size: 12px; cursor: pointer; font-weight: 600;
  }
  .draft-banner { border-radius: 12px; padding: 16px; color: #fff; margin-top: 14px; }
  .tier-RED { background: linear-gradient(135deg, #b91c1c, #dc2626); }
  .tier-ORANGE { background: linear-gradient(135deg, #c2410c, #ea580c); }
  .tier-GREEN { background: linear-gradient(135deg, #15803d, #16a34a); }
  .review-box {
    background: #fffbeb; border: 2px solid #f59e0b; border-radius: 12px;
    padding: 16px; margin-top: 12px;
  }
  .prep-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 10px; margin-top: 8px; }
  @media (max-width: 760px) { .prep-grid { grid-template-columns: 1fr; } }
  .prep-card { background: #fff; border: 1px solid #e2e8f0; border-radius: 8px; padding: 10px; font-size: 13px; }
  .er-card {
    background: #fff; border-radius: 12px; border: 1px solid var(--border);
    border-top: 6px solid var(--red); padding: 16px; margin-bottom: 14px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.06);
  }
  .er-card.ORANGE { border-top-color: var(--orange); }
  .er-card.GREEN { border-top-color: var(--green); }
  .badge { display: inline-block; padding: 4px 10px; border-radius: 999px; font-size: 12px; font-weight: 700; }
  .hidden { display: none !important; }
  table { width: 100%; border-collapse: collapse; font-size: 12.5px; }
  th, td { border-bottom: 1px solid #e2e8f0; padding: 8px; text-align: start; }
</style>
</head>
<body>

<div class="topbar">
  <div class="brand">
    <h1>🚑 RELAY</h1>
    <small id="t_tagline">Bridging the Gap Before Arrival | Pre-Hospital EMS Handover</small>
  </div>
  <div class="nav-actions">
    <button class="btn btn-lang" id="langToggleBtn" onclick="toggleLanguage()">العربية</button>
    <button class="btn btn-er" id="switchViewBtn" onclick="toggleView()">🏥 ER Dashboard</button>
  </div>
</div>

<!-- ================= VIEW 0: EMS LOGIN SCREEN (SCENE 0 & 1) ================= -->
<div id="loginView" class="login-wrap">
  <div class="login-card">
    <div style="text-align:center; margin-bottom:18px;">
      <div style="font-size:38px;">🚑</div>
      <h2 style="margin:6px 0 4px; color:#0f2942;" id="t_login_title">EMS Team Sign In — RELAY</h2>
      <div style="font-size:12.5px; color:#64748b;" id="t_login_sub">Pre-Hospital Emergency Triage & Handover System</div>
    </div>
    <form onsubmit="handleLogin(event)">
      <div style="margin-bottom:12px;">
        <label id="t_login_id_lbl">Paramedic Code (ID):</label>
        <input id="loginCode" placeholder="e.g. EMS001" required autocomplete="off" />
      </div>
      <div style="margin-bottom:16px;">
        <label id="t_login_pin_lbl">Security PIN:</label>
        <input id="loginPin" type="password" placeholder="••••" required />
      </div>
      <div id="loginErr" style="color:#dc2626; font-size:12.5px; margin-bottom:10px; display:none;"></div>
      <button type="submit" class="btn btn-primary" id="t_login_btn">Sign In to Workspace</button>
    </form>
  </div>
</div>

<!-- ================= VIEW 1: PARAMEDIC WORKSPACE ================= -->
<div id="paramedicView" class="container hidden">
  <div class="card" style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;">
    <div>
      <strong id="t_header_title">🚑 New Patient Handover</strong> &nbsp;|&nbsp;
      <span id="t_paramedic_lbl">Logged Paramedic:</span> <b id="loggedParamedicName">Paramedic 1 (EMS001)</b>
    </div>
    <div style="display:flex; gap:8px;">
      <button class="btn btn-outline" style="color:#dc2626; border-color:#fecaca;" id="t_clear_btn" onclick="resetCase()">🗑️ Clear Patient Case</button>
      <button class="btn btn-outline" id="t_logout_btn" onclick="logoutUser()">Sign Out</button>
    </div>
  </div>

  <div class="disclaimer" id="t_disclaimer">
    ⚠️ <b>Demo version — not a clinical decision tool.</b> AI suggests a draft only and may make errors; clinical verification and confirmation remain the responsibility of the attending paramedic.
  </div>

  <div class="status-strip">
    <span id="t_conn_status">✅ RELAY Connected | Active Models: gemini-2.5-flash → gemini-2.5-flash-lite</span>
    <span id="liveSyncNote"></span>
  </div>

  <div class="voice-box">
    <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;">
      <div>
        <strong id="t_voice_title">🎙️ Live Voice Transcription + Real-Time AI Field Population</strong>
        <div id="t_voice_sub" style="font-size:12.5px; color:#0369a1; margin-top:3px;">
          Speak naturally in English or Egyptian Arabic: RELAY transcribes live, populates fields while you speak, and preserves the verbatim transcript.
        </div>
      </div>
      <div style="display:flex; gap:8px; align-items:center;">
        <select id="speechLang" style="width:auto;">
          <option value="en-US">English</option>
          <option value="ar-EG">عربي (مصر)</option>
        </select>
        <button class="btn btn-primary" style="width:auto;" id="recordBtn" onclick="toggleRecording()">🎤 Start recording</button>
      </div>
    </div>
    <div class="interim-box" id="liveTranscriptBox">
      <span id="t_interim_placeholder">Live speech monitor ready — click "Start recording" and speak your handover report...</span>
    </div>
  </div>

  <div class="card">
    <h3 id="t_sec1">👤 1. Patient Identification</h3>
    <div class="grid-4">
      <div>
        <label id="t_lbl_name">Patient Name:</label>
        <input id="pName" placeholder="Enter full name" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
      <div>
        <label id="t_lbl_age">Age (years):</label>
        <input id="pAge" type="number" min="0" max="125" placeholder="Age (0 for infant)" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
      <div>
        <label id="t_lbl_gender">Gender:</label>
        <select id="pGender" onchange="markEdited(this)">
          <option value="" id="opt_g_unspec">Unspecified</option>
          <option value="Male" id="opt_g_male">Male</option>
          <option value="Female" id="opt_g_female">Female</option>
        </select>
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
      <div>
        <label id="t_lbl_nid">National ID (14 digits):</label>
        <input id="pNid" maxlength="14" placeholder="14 digits (optional / masked)" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
    </div>
  </div>

  <div class="card">
    <h3 id="t_sec2">🩺 2. Vital Signs & Mean Arterial Pressure (MAP)</h3>
    <div class="grid-4" style="margin-bottom:12px;">
      <div>
        <label id="t_lbl_sbp">🩸 Systolic BP (SBP mmHg):</label>
        <input id="vSbp" type="number" placeholder="e.g. 120" oninput="markEdited(this); updateMap()" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
      <div>
        <label id="t_lbl_dbp">🩸 Diastolic BP (DBP mmHg):</label>
        <input id="vDbp" type="number" placeholder="e.g. 80" oninput="markEdited(this); updateMap()" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
      <div class="map-box">
        <div style="font-size:12px; font-weight:700;" id="t_map_title">Mean Arterial Pressure (MAP)</div>
        <div class="map-val" id="mapDisplay">--</div>
        <div style="font-size:11px; color:#64748b;">DBP + (SBP - DBP) / 3</div>
      </div>
      <div>
        <label id="t_lbl_hr">💓 Heart Rate HR (bpm):</label>
        <input id="vHr" type="number" placeholder="e.g. 85" oninput="markEdited(this); updateMap()" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
    </div>
    <div class="grid-3">
      <div>
        <label id="t_lbl_spo2">🫁 Oxygen Saturation SpO₂ (%):</label>
        <input id="vSpo2" type="number" placeholder="e.g. 98" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
      <div>
        <label id="t_lbl_rr">🫁 Respiratory Rate RR (breaths/min):</label>
        <input id="vRr" type="number" placeholder="e.g. 16" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
      <div>
        <label id="t_lbl_temp">🌡️ Temperature (°C):</label>
        <input id="vTemp" type="number" step="0.1" placeholder="e.g. 37.0" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
    </div>
  </div>

  <div class="card">
    <h3 id="t_sec3">🧠 3. Consciousness Assessment (AVPU & GCS)</h3>
    <div class="grid-2">
      <div>
        <label id="t_lbl_avpu">AVPU Response Scale:</label>
        <select id="vAvpu" onchange="markEdited(this)">
          <option value="" id="opt_avpu_none">Not assessed</option>
          <option value="A" id="opt_avpu_a">A — Alert</option>
          <option value="V" id="opt_avpu_v">V — Responds to Voice</option>
          <option value="P" id="opt_avpu_p">P — Responds to Pain</option>
          <option value="U" id="opt_avpu_u">U — Unresponsive</option>
        </select>
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
      <div>
        <label id="t_lbl_gcs">Glasgow Coma Scale GCS (3–15):</label>
        <input id="vGcs" type="number" min="3" max="15" placeholder="Enter score (3–15)" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ edited manually</div>
      </div>
    </div>
  </div>

  <div class="card">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;">
      <h3 style="margin:0; border:none; padding:0;" id="t_sec4">📝 4. Chief Complaint & Current Medications</h3>
      <button class="btn btn-dark" id="deepExtractBtn" onclick="runExtraction('deep')">✨ Smart AI Extract</button>
    </div>
    <label id="t_lbl_complaint">Chief Complaint (Verbatim Raw Transcript):</label>
    <textarea id="vComplaint" rows="3" placeholder="Speak above or type clinical handover report here..."></textarea>
    <div class="warn-transcript" id="transcriptWarn">
      ⚠️ Transcript generated from voice recognition and may contain errors — please verify numbers and medication names before confirming.
    </div>
    <div style="margin-top:10px;">
      <label id="t_lbl_meds">💊 Current Medications:</label>
      <input id="vMeds" placeholder="e.g. Aspirin, Metformin, Plavix, Unknown blood thinners" oninput="markEdited(this)" />
      <div class="edit-tag" data-edit-tag>✎ edited manually</div>
    </div>
  </div>

  <div class="card">
    <h3 id="t_sec5">⏱️ 5. Estimated Time of Arrival (ETA)</h3>
    <div style="display:flex; gap:10px; align-items:center; flex-wrap:wrap;">
      <input id="vEta" type="number" min="1" max="180" placeholder="Minutes (1–180)" style="max-width:220px;" oninput="markEdited(this)" />
      <button class="chip" onclick="setEta(5)">5m</button>
      <button class="chip" onclick="setEta(10)">10m</button>
      <button class="chip" onclick="setEta(15)">15m</button>
    </div>
    <div style="margin-top:14px;">
      <button class="btn btn-primary" id="genDraftBtn" onclick="generateDraft()">📝 Create draft for review</button>
    </div>
  </div>

  <!-- 6. Human-in-the-Loop Review Panel (Scene 4 & 5) -->
  <div id="draftContainer" class="hidden">
    <div id="draftBanner" class="draft-banner tier-RED">
      <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px;">
        <strong id="draftTierTitle" style="font-size:17px;">🔴 AI Preliminary Draft: RED Tier</strong>
        <span class="badge" style="background:rgba(0,0,0,0.28);" id="draftScoreBadge">10 / 10</span>
      </div>
      <div style="margin-top:6px; font-size:13px;" id="draftMetaLine"></div>
      <ul id="draftAlertsList" style="margin:10px 0 0; padding-inline-start:20px; font-size:13px;"></ul>
    </div>

    <div class="review-box">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;">
        <strong style="color:#92400e; font-size:15px;" id="t_hitl_title">✋ Review before saving and sending</strong>
        <span style="font-size:12px; color:#78350f;" id="draftModelBadge"></span>
      </div>

      <label id="t_hitl_sum_lbl">Suggested Clinical Summary (Editable):</label>
      <textarea id="rSummary" rows="2" oninput="markEdited(this)"></textarea>
      <div class="edit-tag" data-edit-tag>✎ edited manually</div>

      <div class="grid-3" style="margin-top:10px;">
        <div>
          <label id="t_hitl_tier_lbl">Triage Tier:</label>
          <select id="rTier" onchange="checkDowngradeReason()">
            <option value="RED" id="opt_t_red">🔴 Red — Immediate Resuscitation (9–10)</option>
            <option value="ORANGE" id="opt_t_orange">🟠 Orange — Urgent (5–8)</option>
            <option value="GREEN" id="opt_t_green">🟢 Green — Routine (1–4)</option>
          </select>
        </div>
        <div>
          <label id="t_hitl_score_lbl">Severity Score (1–10):</label>
          <input id="rScore" type="number" min="1" max="10" oninput="markEdited(this)" />
        </div>
        <div>
          <label id="t_hitl_dept_lbl">Receiving Department:</label>
          <input id="rDept" oninput="markEdited(this)" />
        </div>
      </div>

      <div id="downgradeBox" class="hidden" style="margin-top:10px; background:#fef2f2; border:1px solid #fca5a5; padding:10px; border-radius:8px;">
        <label style="color:#991b1b;">⚠️ Mandatory reason for lowering severity below vital-signs floor:</label>
        <input id="downgradeReason" placeholder="Enter clinical justification..." />
      </div>

      <div style="margin-top:12px;">
        <label id="t_hitl_prep_lbl">Suggested Preparation Checklist (Approved Clinical List):</label>
        <div class="prep-grid" id="prepGrid"></div>
      </div>

      <div style="margin-top:14px; background:#fff; padding:10px 12px; border-radius:8px; border:1px solid #fcd34d;">
        <label style="display:flex; gap:8px; align-items:center; cursor:pointer; margin:0;">
          <input type="checkbox" id="hitlConfirmCheck" style="width:18px; height:18px;" />
          <span id="t_hitl_ack">I reviewed the complaint, vitals and suggestions, and I take responsibility for confirming this handover per facility protocol.</span>
        </label>
      </div>

      <div style="display:flex; gap:10px; margin-top:12px;">
        <button class="btn btn-dark" style="flex:2; padding:12px; font-size:14.5px;" id="confirmSendBtn" onclick="confirmAndSend()">✅ Confirm, save and send alert</button>
        <button class="btn btn-outline" style="flex:1;" id="cancelDraftBtn" onclick="document.getElementById('draftContainer').classList.add('hidden')">Cancel draft</button>
      </div>
    </div>
  </div>

  <div class="card" style="margin-top:16px;">
    <h3 id="t_log_title">📋 Confirmed & Saved Handover Log (PHI Masked)</h3>
    <div style="overflow-x:auto;">
      <table>
        <thead>
          <tr>
            <th>Patient ID</th><th>Tier</th><th>Patient</th><th>Masked ID</th><th>Vitals / MAP</th><th>ETA</th><th>Edited</th>
          </tr>
        </thead>
        <tbody id="auditTableBody"></tbody>
      </table>
    </div>
  </div>
</div>

<!-- ================= VIEW 2: ER DASHBOARD (SCENE 6) ================= -->
<div id="erView" class="container hidden">
  <div class="card" style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;">
    <div>
      <strong style="font-size:17px;" id="t_er_title">🏥 ER Live Dashboard</strong>
      <div style="font-size:12.5px; color:#475569;" id="t_er_sub">Real-time pre-arrival telemetry & resuscitation room readiness</div>
    </div>
    <div style="display:flex; gap:8px; align-items:center;">
      <span class="badge" style="background:#dcfce7; color:#166534;" id="t_er_live">● Live Sync (3s)</span>
      <button class="btn btn-outline" id="chimeBtn" onclick="toggleChime()">🔔 Audio Alert: ON</button>
    </div>
  </div>

  <div class="grid-4" style="margin-bottom:14px;">
    <div class="card" style="text-align:center; border-top:4px solid #0284c7;">
      <div style="font-size:12px; color:#475569;">Total Incoming</div>
      <div style="font-size:26px; font-weight:800;" id="cntTotal">0</div>
    </div>
    <div class="card" style="text-align:center; border-top:4px solid #dc2626;">
      <div style="font-size:12px; color:#dc2626;">🔴 Red</div>
      <div style="font-size:26px; font-weight:800; color:#dc2626;" id="cntRed">0</div>
    </div>
    <div class="card" style="text-align:center; border-top:4px solid #ea580c;">
      <div style="font-size:12px; color:#ea580c;">🟠 Orange</div>
      <div style="font-size:26px; font-weight:800; color:#ea580c;" id="cntOrange">0</div>
    </div>
    <div class="card" style="text-align:center; border-top:4px solid #16a34a;">
      <div style="font-size:12px; color:#16a34a;">🟢 Green</div>
      <div style="font-size:26px; font-weight:800; color:#16a34a;" id="cntGreen">0</div>
    </div>
  </div>

  <div id="erCasesList"></div>
</div>

<script>
  let uiLang = 'en';
  let isLoggedIn = false;
  let currentParamedic = 'Paramedic 1 (EMS001)';
  let currentView = window.location.pathname === '/er' ? 'er' : 'workspace';
  let isRecording = false;
  let recognition = null;
  let mediaRecorder = null;
  let audioChunks = [];
  let liveExtractTimer = null;
  let lastExtractedSnapshot = '';
  let currentDraft = null;
  let anyManualEdit = false;
  let chimeEnabled = true;
  let knownCaseIds = new Set();

  const VALID_CREDS = { 'EMS001': '2580', 'EMS002': '1470' };

  function handleLogin(e) {
    e.preventDefault();
    const code = document.getElementById('loginCode').value.trim().toUpperCase();
    const pin = document.getElementById('loginPin').value.trim();
    const err = document.getElementById('loginErr');
    if (VALID_CREDS[code] && VALID_CREDS[code] === pin) {
      isLoggedIn = true;
      currentParamedic = `Paramedic (${code})`;
      document.getElementById('loggedParamedicName').textContent = currentParamedic;
      err.style.display = 'none';
      document.getElementById('loginView').classList.add('hidden');
      document.getElementById('paramedicView').classList.remove('hidden');
    } else {
      err.style.display = 'block';
      err.textContent = uiLang === 'en' ? 'Invalid Paramedic Code or PIN (Use EMS001 / 2580).' : 'كود المسعف أو الرقم السري غير صحيح (استخدم EMS001 / 2580).';
    }
  }

  function logoutUser() {
    isLoggedIn = false;
    document.getElementById('paramedicView').classList.add('hidden');
    document.getElementById('loginView').classList.remove('hidden');
    document.getElementById('loginPin').value = '';
  }

  const I18N = {
    en: {
      dir: 'ltr', langToggle: 'العربية',
      switchToEr: '🏥 ER Dashboard', switchToParam: '🚑 Paramedic Workspace',
      tagline: 'Bridging the Gap Before Arrival | Pre-Hospital EMS Handover',
      loginTitle: 'EMS Team Sign In — RELAY', loginSub: 'Pre-Hospital Emergency Triage & Handover System',
      loginId: 'Paramedic Code (ID):', loginPin: 'Security PIN:', loginBtn: 'Sign In to Workspace',
      headerTitle: '🚑 New Patient Handover', paramedicLbl: 'Logged Paramedic:',
      clearBtn: '🗑️ Clear Patient Case', logoutBtn: 'Sign Out',
      disclaimer: '⚠️ <b>Demo version — not a clinical decision tool.</b> AI suggests a draft only and may make errors; clinical verification and confirmation remain the responsibility of the attending paramedic.',
      connStatus: '✅ RELAY Connected | Active Models: gemini-2.5-flash → gemini-2.5-flash-lite',
      voiceTitle: '🎙️ Live Voice Transcription + Real-Time AI Field Population',
      voiceSub: 'Speak naturally in English or Egyptian Arabic: RELAY transcribes live, populates fields while you speak, and preserves the verbatim transcript.',
      recStart: '🎤 Start recording', recStop: '⏹️ Stop recording',
      interimReady: 'Live speech monitor ready — click "Start recording" and speak your handover report...',
      sec1: '👤 1. Patient Identification', lblName: 'Patient Name:', phName: 'Enter full name',
      lblAge: 'Age (years):', phAge: 'Age (0 for infant)', lblGender: 'Gender:',
      gUnspec: 'Unspecified', gMale: 'Male', gFemale: 'Female',
      lblNid: 'National ID (14 digits):', phNid: '14 digits (optional / masked)',
      sec2: '🩺 2. Vital Signs & Mean Arterial Pressure (MAP)',
      lblSbp: '🩸 Systolic BP (SBP mmHg):', lblDbp: '🩸 Diastolic BP (DBP mmHg):',
      mapTitle: 'Mean Arterial Pressure (MAP)', lblHr: '💓 Heart Rate HR (bpm):',
      lblSpo2: '🫁 Oxygen Saturation SpO₂ (%):', lblRr: '🫁 Respiratory Rate RR (breaths/min):',
      lblTemp: '🌡️ Temperature (°C):',
      sec3: '🧠 3. Consciousness Assessment (AVPU & GCS)',
      lblAvpu: 'AVPU Response Scale:', avpuNone: 'Not assessed', avpuA: 'A — Alert',
      avpuV: 'V — Responds to Voice', avpuP: 'P — Responds to Pain', avpuU: 'U — Unresponsive',
      lblGcs: 'Glasgow Coma Scale GCS (3–15):', phGcs: 'Enter score (3–15)',
      sec4: '📝 4. Chief Complaint & Current Medications',
      deepBtn: '✨ Smart AI Extract',
      lblComplaint: 'Chief Complaint (Verbatim Raw Transcript):',
      phComplaint: 'Speak above or type clinical handover report here...',
      lblMeds: '💊 Current Medications:',
      phMeds: 'e.g. Aspirin, Metformin, Plavix, Unknown blood thinners',
      sec5: '⏱️ 5. Estimated Time of Arrival (ETA)', phEta: 'Minutes (1–180)',
      genBtn: '📝 Create draft for review',
      hitlTitle: '✋ Review before saving and sending',
      hitlSumLbl: 'Suggested Clinical Summary (Editable):',
      hitlTierLbl: 'Triage Tier:', hitlScoreLbl: 'Severity Score (1–10):',
      hitlDeptLbl: 'Receiving Department:',
      tRed: '🔴 Red — Immediate Resuscitation (9–10)',
      tOrange: '🟠 Orange — Urgent (5–8)',
      tGreen: '🟢 Green — Routine (1–4)',
      hitlPrepLbl: 'Suggested Preparation Checklist (Approved Clinical List):',
      hitlAck: 'I reviewed the complaint, vitals and suggestions, and I take responsibility for confirming this handover per facility protocol.',
      confirmBtn: '✅ Confirm, save and send alert', cancelBtn: 'Cancel draft',
      editTag: '✎ edited manually',
      noCases: '🚑 No incoming cases yet. Confirmed ambulance handovers will appear here instantaneously.',
      prepCats: {
        immediate_actions: '🚑 Immediate Actions', staff_to_alert: '👨‍⚕️ Staff to Alert',
        equipment: '🩺 Equipment', labs_imaging: '🔬 Labs & Imaging',
        medications_prep: '💊 Medications Prep', precautions: '⚠️ Precautions'
      }
    },
    ar: {
      dir: 'rtl', langToggle: 'English',
      switchToEr: '🏥 شاشة استقبال الطوارئ (ER Dashboard)', switchToParam: '🚑 واجهة تسجيل المسعف',
      tagline: 'سد الفجوة قبل الوصول | تسليم حالات الطوارئ ما قبل المستشفى',
      loginTitle: 'دخول طاقم الإسعاف — RELAY', loginSub: 'نظام الفرز والتسليم المسبق لحالات الطوارئ',
      loginId: 'كود المسعف:', loginPin: 'الرقم السري (PIN):', loginBtn: 'دخول إلى النظام',
      headerTitle: '🚑 تسجيل حالة جديدة', paramedicLbl: 'المسعف المسجل:',
      clearBtn: '🗑️ مسح بيانات الحالة', logoutBtn: 'تسجيل الخروج',
      disclaimer: '⚠️ <b>نسخة تجريبية — ليست أداة قرار سريري.</b> الذكاء الاصطناعي يقترح مسودة فقط وقد يخطئ، والقرار والتأكيد للمسعف المسؤول.',
      connStatus: '✅ RELAY متصل | النماذج: gemini-2.5-flash → gemini-2.5-flash-lite',
      voiceTitle: '🎙️ التسجيل الصوتي المباشر + التعبئة الذكية الفورية',
      voiceSub: 'تحدث بالعربية أو الإنجليزية: يقوم النظام بالتفريغ الحي وتعبئة الخانات مع حفظ النص الحرفي.',
      recStart: '🎤 ابدأ التسجيل', recStop: '⏹️ إيقاف التسجيل',
      interimReady: 'مراقب الصوت جاهز — اضغط "ابدأ التسجيل" وتحدث...',
      sec1: '👤 1. بيانات تعريف المريض', lblName: 'اسم المريض:', phName: 'أدخل الاسم',
      lblAge: 'العمر (سنة):', phAge: 'العمر', lblGender: 'النوع:',
      gUnspec: 'غير محدد', gMale: 'ذكر', gFemale: 'أنثى',
      lblNid: 'الرقم القومي (14 رقماً):', phNid: '14 رقماً',
      sec2: '🩺 2. العلامات الحيوية ومتوسط الضغط الشرياني (MAP)',
      lblSbp: '🩸 الضغط الانقباضي (SBP):', lblDbp: '🩸 الضغط الانبساطي (DBP):',
      mapTitle: 'متوسط الضغط الشرياني (MAP)', lblHr: '💓 النبض HR:',
      lblSpo2: '🫁 الأكسجين SpO₂ (%):', lblRr: '🫁 معدل التنفس RR:',
      lblTemp: '🌡️ الحرارة (°C):',
      sec3: '🧠 3. تقييم مستوى الوعي (AVPU & GCS)',
      lblAvpu: 'مقياس AVPU:', avpuNone: 'لم يُقيّم', avpuA: 'A — واعٍ',
      avpuV: 'V — يستجيب للصوت', avpuP: 'P — يستجيب للألم', avpuU: 'U — غير مستجيب',
      lblGcs: 'مقياس غلاسكو GCS (3–15):', phGcs: 'الدرجة',
      sec4: '📝 4. الشكوى الرئيسية والأدوية',
      deepBtn: '✨ استخراج ذكي من النص',
      lblComplaint: 'الشكوى الرئيسية (النص الحرفي):',
      phComplaint: 'تحدث بالأعلى أو اكتب الحالة...',
      lblMeds: '💊 الأدوية الحالية:',
      phMeds: 'الأدوية الحالية',
      sec5: '⏱️ 5. وقت الوصول المتوقع (ETA)', phEta: 'الدقائق',
      genBtn: '📝 إنشاء مسودة للمراجعة',
      hitlTitle: '✋ مراجعة قبل الحفظ والإرسال',
      hitlSumLbl: 'الملخص المقترح (قابل للتعديل):',
      hitlTierLbl: 'الفئة:', hitlScoreLbl: 'الدرجة (1–10):',
      hitlDeptLbl: 'القسم:',
      tRed: '🔴 أحمر (9–10)', tOrange: '🟠 برتقالي (5–8)', tGreen: '🟢 أخضر (1–4)',
      hitlPrepLbl: 'التجهيزات المقترحة:',
      hitlAck: 'راجعت الشكوى والعلامات الحيوية والاقتراحات، وأتحمل مسؤولية تأكيد بيانات التسليم.',
      confirmBtn: '✅ تأكيد وحفظ وإرسال التنبيه', cancelBtn: 'إلغاء المسودة',
      editTag: '✎ تم التعديل يدوياً',
      noCases: '🚑 لا توجد حالات واردة حالياً.',
      prepCats: {
        immediate_actions: '🚑 إجراءات فورية', staff_to_alert: '👨‍⚕️ تخصصات يجب إخطارها',
        equipment: '🩺 معدات وأجهزة', labs_imaging: '🔬 فحوصات وأشعة',
        medications_prep: '💊 أدوية للتجهيز', precautions: '⚠️ احتياطات'
      }
    }
  };

  function applyLanguage() {
    const t = I18N[uiLang];
    document.documentElement.lang = uiLang;
    document.documentElement.dir = t.dir;
    document.getElementById('langToggleBtn').textContent = t.langToggle;
    document.getElementById('switchViewBtn').textContent = currentView === 'er' ? t.switchToParam : t.switchToEr;
    document.getElementById('t_tagline').textContent = t.tagline;

    document.getElementById('t_login_title').textContent = t.loginTitle;
    document.getElementById('t_login_sub').textContent = t.loginSub;
    document.getElementById('t_login_id_lbl').textContent = t.loginId;
    document.getElementById('t_login_pin_lbl').textContent = t.loginPin;
    document.getElementById('t_login_btn').textContent = t.loginBtn;

    document.getElementById('t_header_title').textContent = t.headerTitle;
    document.getElementById('t_paramedic_lbl').textContent = t.paramedicLbl;
    document.getElementById('t_clear_btn').textContent = t.clearBtn;
    document.getElementById('t_logout_btn').textContent = t.logoutBtn;
    document.getElementById('t_disclaimer').innerHTML = t.disclaimer;
    document.getElementById('t_conn_status').textContent = t.connStatus;
    document.getElementById('t_voice_title').textContent = t.voiceTitle;
    document.getElementById('t_voice_sub').textContent = t.voiceSub;
    document.getElementById('recordBtn').textContent = isRecording ? t.recStop : t.recStart;
    if (!isRecording) document.getElementById('liveTranscriptBox').textContent = t.interimReady;
    document.getElementById('speechLang').value = uiLang === 'en' ? 'en-US' : 'ar-EG';

    document.getElementById('t_sec1').textContent = t.sec1;
    document.getElementById('t_lbl_name').textContent = t.lblName;
    document.getElementById('pName').placeholder = t.phName;
    document.getElementById('t_lbl_age').textContent = t.lblAge;
    document.getElementById('pAge').placeholder = t.phAge;
    document.getElementById('t_lbl_gender').textContent = t.lblGender;
    document.getElementById('opt_g_unspec').textContent = t.gUnspec;
    document.getElementById('opt_g_male').textContent = t.gMale;
    document.getElementById('opt_g_female').textContent = t.gFemale;
    document.getElementById('t_lbl_nid').textContent = t.lblNid;
    document.getElementById('pNid').placeholder = t.phNid;

    document.getElementById('t_sec2').textContent = t.sec2;
    document.getElementById('t_lbl_sbp').textContent = t.lblSbp;
    document.getElementById('t_lbl_dbp').textContent = t.lblDbp;
    document.getElementById('t_map_title').textContent = t.mapTitle;
    document.getElementById('t_lbl_hr').textContent = t.lblHr;
    document.getElementById('t_lbl_spo2').textContent = t.lblSpo2;
    document.getElementById('t_lbl_rr').textContent = t.lblRr;
    document.getElementById('t_lbl_temp').textContent = t.lblTemp;

    document.getElementById('t_sec3').textContent = t.sec3;
    document.getElementById('t_lbl_avpu').textContent = t.lblAvpu;
    document.getElementById('opt_avpu_none').textContent = t.avpuNone;
    document.getElementById('opt_avpu_a').textContent = t.avpuA;
    document.getElementById('opt_avpu_v').textContent = t.avpuV;
    document.getElementById('opt_avpu_p').textContent = t.avpuP;
    document.getElementById('opt_avpu_u').textContent = t.avpuU;
    document.getElementById('t_lbl_gcs').textContent = t.lblGcs;
    document.getElementById('vGcs').placeholder = t.phGcs;

    document.getElementById('t_sec4').textContent = t.sec4;
    document.getElementById('deepExtractBtn').textContent = t.deepBtn;
    document.getElementById('t_lbl_complaint').textContent = t.lblComplaint;
    document.getElementById('vComplaint').placeholder = t.phComplaint;
    document.getElementById('t_lbl_meds').textContent = t.lblMeds;
    document.getElementById('vMeds').placeholder = t.phMeds;

    document.getElementById('t_sec5').textContent = t.sec5;
    document.getElementById('vEta').placeholder = t.phEta;
    document.getElementById('genDraftBtn').textContent = t.genBtn;

    document.getElementById('t_hitl_title').textContent = t.hitlTitle;
    document.getElementById('t_hitl_sum_lbl').textContent = t.hitlSumLbl;
    document.getElementById('t_hitl_tier_lbl').textContent = t.hitlTierLbl;
    document.getElementById('t_hitl_score_lbl').textContent = t.hitlScoreLbl;
    document.getElementById('t_hitl_dept_lbl').textContent = t.hitlDeptLbl;
    document.getElementById('opt_t_red').textContent = t.tRed;
    document.getElementById('opt_t_orange').textContent = t.tOrange;
    document.getElementById('opt_t_green').textContent = t.tGreen;
    document.getElementById('t_hitl_prep_lbl').textContent = t.hitlPrepLbl;
    document.getElementById('t_hitl_ack').textContent = t.hitlAck;
    document.getElementById('confirmSendBtn').textContent = t.confirmBtn;
    document.getElementById('cancelDraftBtn').textContent = t.cancelBtn;
    document.querySelectorAll('[data-edit-tag]').forEach(el => el.textContent = t.editTag);
    refreshCases();
  }

  function toggleLanguage() {
    uiLang = uiLang === 'en' ? 'ar' : 'en';
    applyLanguage();
  }

  function toggleView() {
    if (currentView !== 'er') {
      currentView = 'er';
      document.getElementById('loginView').classList.add('hidden');
      document.getElementById('paramedicView').classList.add('hidden');
      document.getElementById('erView').classList.remove('hidden');
    } else {
      currentView = 'workspace';
      document.getElementById('erView').classList.add('hidden');
      if (isLoggedIn) document.getElementById('paramedicView').classList.remove('hidden');
      else document.getElementById('loginView').classList.remove('hidden');
    }
    applyLanguage();
  }

  async function toggleRecording() {
    if (isRecording) stopRecording();
    else await startRecording();
  }

  async function startRecording() {
    const t = I18N[uiLang];
    const box = document.getElementById('liveTranscriptBox');
    isRecording = true;
    document.getElementById('recordBtn').textContent = t.recStop;
    document.getElementById('recordBtn').classList.remove('btn-primary');
    document.getElementById('recordBtn').classList.add('btn-danger');
    box.innerHTML = uiLang === 'en' ? '🎙️ <b>Listening live...</b> Speak clearly now.' : '🎙️ <b>جاري الاستماع الحي...</b> تحدث الآن.';

    audioChunks = [];
    try {
      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      mediaRecorder = new MediaRecorder(stream);
      mediaRecorder.ondataavailable = e => { if (e.data.size > 0) audioChunks.push(e.data); };
      mediaRecorder.onstop = async () => {
        stream.getTracks().forEach(tr => tr.stop());
        if (document.getElementById('vComplaint').value.trim().length < 5 && audioChunks.length > 0) {
          box.textContent = uiLang === 'en' ? '⏳ Transcribing recorded audio via Gemini...' : '⏳ جاري التفريغ الصوتي عبر Gemini...';
          const blob = new Blob(audioChunks, { type: mediaRecorder.mimeType || 'audio/webm' });
          const reader = new FileReader();
          reader.onloadend = async () => {
            const b64 = reader.result.split(',')[1];
            const res = await fetch('/api/transcribe_audio', {
              method: 'POST', headers: {'Content-Type': 'application/json'},
              body: JSON.stringify({ audio_b64: b64, mime_type: blob.type, lang: uiLang })
            }).then(r => r.json());
            if (res.ok && res.transcript) {
              document.getElementById('vComplaint').value = res.transcript;
              document.getElementById('transcriptWarn').style.display = 'block';
              box.textContent = '✅ ' + res.transcript;
              runExtraction('deep');
            }
          };
          reader.readAsDataURL(blob);
        }
      };
      mediaRecorder.start();
    } catch (_) {}

    const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
    if (SR) {
      recognition = new SR();
      recognition.lang = document.getElementById('speechLang').value;
      recognition.continuous = true;
      recognition.interimResults = true;
      recognition.onresult = (event) => {
        let interim = '', finalChunk = '';
        for (let i = event.resultIndex; i < event.results.length; i++) {
          const piece = event.results[i][0].transcript;
          if (event.results[i].isFinal) finalChunk += piece + ' ';
          else interim += piece;
        }
        if (finalChunk.trim()) {
          const ta = document.getElementById('vComplaint');
          ta.value = (ta.value + ' ' + finalChunk).trim();
          document.getElementById('transcriptWarn').style.display = 'block';
          scheduleLiveExtraction();
        }
        box.innerHTML = (uiLang === 'en' ? '🎙️ <b>Live:</b> ' : '🎙️ <b>مباشر:</b> ') +
          (interim ? `<i>${interim}</i>` : document.getElementById('vComplaint').value);
      };
      recognition.onend = () => { if (isRecording) { try { recognition.start(); } catch (_) {} } };
      try { recognition.start(); } catch (_) {}
    }
  }

  function stopRecording() {
    isRecording = false;
    const t = I18N[uiLang];
    const btn = document.getElementById('recordBtn');
    btn.textContent = t.recStart;
    btn.classList.remove('btn-danger');
    btn.classList.add('btn-primary');
    if (recognition) { try { recognition.stop(); } catch (_) {} }
    if (mediaRecorder && mediaRecorder.state !== 'inactive') { try { mediaRecorder.stop(); } catch (_) {} }
    if (document.getElementById('vComplaint').value.trim().length >= 5) {
      document.getElementById('transcriptWarn').style.display = 'block';
      runExtraction('deep');
    }
  }

  function scheduleLiveExtraction() {
    clearTimeout(liveExtractTimer);
    liveExtractTimer = setTimeout(() => {
      const txt = document.getElementById('vComplaint').value.trim();
      if (txt.length >= 12 && txt !== lastExtractedSnapshot) {
        lastExtractedSnapshot = txt;
        runExtraction('fast');
      }
    }, 2000);
  }

  function applyFieldIfUnedited(id, val) {
    if (val === null || val === undefined || val === '') return;
    const el = document.getElementById(id);
    if (!el || el.dataset.manual === '1') return;
    el.value = val;
  }

  async function runExtraction(mode) {
    const txt = document.getElementById('vComplaint').value.trim();
    const note = document.getElementById('liveSyncNote');
    if (txt.length < 4) return;
    const btn = document.getElementById('deepExtractBtn');
    if (mode === 'deep') btn.disabled = true;
    note.textContent = uiLang === 'en' ? '⚡ Extracting fields...' : '⚡ جاري استخراج البيانات...';
    try {
      const res = await fetch('/api/extract', {
        method: 'POST', headers: {'Content-Type': 'application/json'},
        body: JSON.stringify({ text: txt, lang: uiLang, mode })
      }).then(r => r.json());
      if (res.ok && res.fields) {
        const f = res.fields;
        applyFieldIfUnedited('pName', f.patient_name);
        applyFieldIfUnedited('pAge', f.age);
        if (f.gender) {
          const g = String(f.gender).toLowerCase();
          applyFieldIfUnedited('pGender', g.startsWith('f') ? 'Female' : (g.startsWith('m') ? 'Male' : ''));
        }
        applyFieldIfUnedited('pNid', f.national_id);
        applyFieldIfUnedited('vSbp', f.sbp);
        applyFieldIfUnedited('vDbp', f.dbp);
        applyFieldIfUnedited('vHr', f.hr);
        applyFieldIfUnedited('vSpo2', f.spo2);
        applyFieldIfUnedited('vRr', f.rr);
        applyFieldIfUnedited('vTemp', f.temp);
        applyFieldIfUnedited('vAvpu', f.avpu);
        applyFieldIfUnedited('vGcs', f.gcs);
        applyFieldIfUnedited('vMeds', f.medications);
        applyFieldIfUnedited('vEta', f.eta);
        updateMap();
        note.textContent = uiLang === 'en' ? `✅ Synced (${res.model})` : `✅ تم التحديث (${res.model})`;
      }
    } finally {
      if (mode === 'deep') btn.disabled = false;
    }
  }

  function markEdited(el) {
    el.dataset.manual = '1';
    el.classList.add('edited');
    anyManualEdit = true;
    const tag = el.parentElement.querySelector('[data-edit-tag]');
    if (tag) tag.style.display = 'block';
  }

  function updateMap() {
    const sbp = parseFloat(document.getElementById('vSbp').value);
    const dbp = parseFloat(document.getElementById('vDbp').value);
    const hr = parseFloat(document.getElementById('vHr').value);
    const disp = document.getElementById('mapDisplay');
    if (!isNaN(sbp) && !isNaN(dbp) && sbp > 0 && dbp > 0) {
      const map = Math.round(dbp + (sbp - dbp) / 3);
      const si = (!isNaN(hr) && hr > 0) ? ` | SI ${(hr / sbp).toFixed(2)}` : '';
      disp.textContent = `${map} mmHg${si}`;
      disp.style.color = map < 65 ? '#dc2626' : '#0f2942';
    } else {
      disp.textContent = '--';
    }
  }

  function setEta(mins) {
    const el = document.getElementById('vEta');
    el.value = mins;
    markEdited(el);
  }

  function resetCase() {
    ['pName','pAge','pGender','pNid','vSbp','vDbp','vHr','vSpo2','vRr','vTemp','vAvpu','vGcs','vComplaint','vMeds','vEta'].forEach(id => {
      const el = document.getElementById(id);
      el.value = '';
      el.dataset.manual = '0';
      el.classList.remove('edited');
    });
    document.querySelectorAll('[data-edit-tag]').forEach(el => el.style.display = 'none');
    document.getElementById('transcriptWarn').style.display = 'none';
    document.getElementById('draftContainer').classList.add('hidden');
    anyManualEdit = false;
    lastExtractedSnapshot = '';
    updateMap();
  }

  function numOrNull(id) {
    const v = document.getElementById(id).value.trim();
    return v === '' ? null : Number(v);
  }

  function checkDowngradeReason() {
    if (!currentDraft) return;
    const order = { 'GREEN': 1, 'ORANGE': 2, 'RED': 3 };
    const orig = currentDraft.vitals_eval.tier;
    const chosen = document.getElementById('rTier').value;
    document.getElementById('downgradeBox').classList.toggle('hidden', order[chosen] >= order[orig]);
  }

  async function generateDraft() {
    const btn = document.getElementById('genDraftBtn');
    btn.disabled = true;
    const payload = {
      lang: uiLang,
      patient_name: document.getElementById('pName').value.trim() || (uiLang === 'en' ? 'Fictional Patient' : 'مريض تجريبي'),
      age: numOrNull('pAge'),
      gender: document.getElementById('pGender').value || 'Male',
      national_id: document.getElementById('pNid').value.trim(),
      sbp: numOrNull('vSbp'), dbp: numOrNull('vDbp'), hr: numOrNull('vHr'),
      spo2: numOrNull('vSpo2'), rr: numOrNull('vRr'), temp: numOrNull('vTemp'),
      avpu: document.getElementById('vAvpu').value, gcs: numOrNull('vGcs'),
      complaint: document.getElementById('vComplaint').value.trim(),
      medications: document.getElementById('vMeds').value.trim(),
      eta: numOrNull('vEta') || 6
    };
    try {
      const res = await fetch('/api/draft', {
        method: 'POST', headers: {'Content-Type': 'application/json'},
        body: JSON.stringify(payload)
      }).then(r => r.json());
      if (res.ok) {
        currentDraft = { ...payload, ...res };
        const ve = res.vitals_eval, dr = res.draft;
        document.getElementById('draftBanner').className = `draft-banner tier-${ve.tier}`;
        document.getElementById('draftTierTitle').textContent = `🔴 AI Preliminary Draft: ${ve.tier} TIER`;
        document.getElementById('draftScoreBadge').textContent = `${ve.score} / 10`;
        document.getElementById('draftMetaLine').textContent = `Patient ID: ${res.patient_id} | MAP: ${ve.map ?? '--'} mmHg | Shock Index: ${ve.shock_index ?? '--'}`;
        document.getElementById('draftAlertsList').innerHTML = ve.alerts.map(a => `<li>${a}</li>`).join('');
        document.getElementById('draftModelBadge').textContent = `🤖 ${res.model}`;
        document.getElementById('rSummary').value = dr.clinical_summary;
        document.getElementById('rTier').value = ve.tier;
        document.getElementById('rScore').value = ve.score;
        document.getElementById('rDept').value = dr.suggested_department;
        document.getElementById('hitlConfirmCheck').checked = false;
        document.getElementById('downgradeBox').classList.add('hidden');
        const cats = I18N[uiLang].prepCats;
        document.getElementById('prepGrid').innerHTML = Object.keys(cats).map(k =>
          `<div class="prep-card"><strong>${cats[k]}</strong><div style="margin-top:4px; color:#334155;">${(dr[k] || []).join(' • ') || '-'}</div></div>`
        ).join('');
        document.getElementById('draftContainer').classList.remove('hidden');
        document.getElementById('draftContainer').scrollIntoView({ behavior: 'smooth' });
      }
    } finally {
      btn.disabled = false;
      btn.textContent = I18N[uiLang].genBtn;
    }
  }

  async function confirmAndSend() {
    if (!document.getElementById('hitlConfirmCheck').checked) {
      alert(uiLang === 'en' ? 'Please confirm clinical responsibility before sending.' : 'يرجى تحديد مربع تأكيد المسؤولية.');
      return;
    }
    const order = { 'GREEN': 1, 'ORANGE': 2, 'RED': 3 };
    if (order[document.getElementById('rTier').value] < order[currentDraft.vitals_eval.tier] && !document.getElementById('downgradeReason').value.trim()) {
      alert(uiLang === 'en' ? 'Mandatory: provide a written reason for lowering severity below vital signs floor.' : 'إلزامي: اكتب سبب تخفيض الفئة.');
      return;
    }
    const record = {
      lang: uiLang,
      patient_id: currentDraft.patient_id,
      patient_name: currentDraft.patient_name,
      age: currentDraft.age, gender: currentDraft.gender,
      national_id: currentDraft.national_id,
      sbp: currentDraft.sbp, dbp: currentDraft.dbp,
      map: currentDraft.vitals_eval.map, shock_index: currentDraft.vitals_eval.shock_index,
      hr: currentDraft.hr, spo2: currentDraft.spo2, rr: currentDraft.rr, temp: currentDraft.temp,
      avpu: currentDraft.avpu, gcs: currentDraft.gcs,
      medications: currentDraft.medications, eta: currentDraft.eta,
      tier: document.getElementById('rTier').value,
      score: Number(document.getElementById('rScore').value),
      department: document.getElementById('rDept').value,
      summary: document.getElementById('rSummary').value,
      prep: currentDraft.draft,
      manually_edited: anyManualEdit,
      paramedic: currentParamedic,
      model: currentDraft.model
    };
    const res = await fetch('/api/confirm', {
      method: 'POST', headers: {'Content-Type': 'application/json'},
      body: JSON.stringify(record)
    }).then(r => r.json());
    if (res.ok) {
      document.getElementById('draftContainer').classList.add('hidden');
      await refreshCases();
    }
  }

  function playChime() {
    if (!chimeEnabled) return;
    try {
      const ctx = new (window.AudioContext || window.webkitAudioContext)();
      const osc = ctx.createOscillator();
      osc.type = 'sine';
      osc.frequency.setValueAtTime(880, ctx.currentTime);
      osc.connect(ctx.destination);
      osc.start();
      osc.stop(ctx.currentTime + 0.3);
    } catch (_) {}
  }

  function toggleChime() {
    chimeEnabled = !chimeEnabled;
    document.getElementById('chimeBtn').textContent = `🔔 Audio Alert: ${chimeEnabled ? 'ON' : 'OFF'}`;
  }

  async function refreshCases() {
    try {
      const cases = await fetch('/api/cases').then(r => r.json());
      let isNew = false;
      cases.forEach(c => {
        if (!knownCaseIds.has(c.patient_id)) {
          if (knownCaseIds.size > 0 || currentView === 'er') isNew = true;
          knownCaseIds.add(c.patient_id);
        }
      });
      if (isNew && currentView === 'er') playChime();

      document.getElementById('auditTableBody').innerHTML = cases.map(c => `
        <tr>
          <td><code>${c.patient_id}</code></td>
          <td><b>${c.tier}</b> (${c.score}/10)</td>
          <td>${c.patient_name} (${c.age ?? '?'}y, ${c.gender})</td>
          <td><code>${c.masked_nid}</code></td>
          <td>${c.sbp ?? '-'}/${c.dbp ?? '-'} (MAP ${c.map ?? '-'}) | HR ${c.hr ?? '-'}</td>
          <td>${c.eta}m</td>
          <td>${c.manually_edited ? '✎ Yes' : 'No'}</td>
        </tr>
      `).join('');

      document.getElementById('cntTotal').textContent = cases.length;
      document.getElementById('cntRed').textContent = cases.filter(c => c.tier === 'RED').length;
      document.getElementById('cntOrange').textContent = cases.filter(c => c.tier === 'ORANGE').length;
      document.getElementById('cntGreen').textContent = cases.filter(c => c.tier === 'GREEN').length;

      const list = document.getElementById('erCasesList');
      if (!cases.length) {
        list.innerHTML = `<div class="card" style="text-align:center; padding:34px; color:#64748b;">${I18N[uiLang].noCases}</div>`;
        return;
      }
      const cats = I18N[uiLang].prepCats;
      list.innerHTML = cases.map(c => {
        const badgeBg = c.tier === 'RED' ? '#fee2e2; color:#991b1b' : (c.tier === 'ORANGE' ? '#ffedd5; color:#9a3412' : '#dcfce7; color:#166534');
        const prepHtml = Object.keys(cats).map(k => {
          const arr = (c.prep && c.prep[k]) || [];
          return arr.length ? `<div style="font-size:12.5px; margin-top:4px;"><b>${cats[k]}:</b> ${arr.join(' • ')}</div>` : '';
        }).join('');
        return `
          <div class="er-card ${c.tier}">
            <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px;">
              <div>
                <span class="badge" style="background:${badgeBg};">${c.tier} — ${c.score}/10</span>
                <code style="margin-inline-start:8px; font-weight:700; color:#0284c7;">${c.patient_id}</code>
              </div>
              <span class="badge" style="background:#0f172a; color:#fff;">⏱️ ETA: ${c.eta} min</span>
            </div>
            <div style="margin-top:10px; font-size:14px;">
              <b>${c.patient_name}</b> | ${c.age ?? '?'} yrs | ${c.gender} |
              <span>National ID: <code>${c.masked_nid}</code></span> |
              <span>Dept: <b>${c.department}</b></span>
            </div>
            <div style="margin-top:8px; background:#f8fafc; padding:8px 12px; border-radius:8px; font-size:13px;">
              <b>BP:</b> ${c.sbp ?? '-'}/${c.dbp ?? '-'} mmHg (<b>MAP:</b> ${c.map ?? '-'} mmHg | <b>SI:</b> ${c.shock_index ?? '-'}) |
              <b>HR:</b> ${c.hr ?? '-'} bpm | <b>SpO₂:</b> ${c.spo2 ?? '-'}% | <b>RR:</b> ${c.rr ?? '-'}/m | <b>Temp:</b> ${c.temp ?? '-'}°C |
              <b>AVPU:</b> ${c.avpu || 'A'} | <b>GCS:</b> ${c.gcs ?? '-'}
            </div>
            <div style="margin-top:8px; font-size:13.5px;"><b>Summary:</b> ${c.summary}</div>
            ${c.medications ? `<div style="margin-top:6px; background:#fef2f2; color:#991b1b; padding:6px 10px; border-radius:6px; font-size:12.5px;">💊 <b>Medications / Interaction Alert:</b> ${c.medications}</div>` : ''}
            <div style="margin-top:8px; border-top:1px solid #e2e8f0; padding-top:8px;">${prepHtml}</div>
          </div>
        `;
      }).join('');
    } catch (_) {}
  }

  setInterval(refreshCases, 3000);
  if (currentView === 'er') {
    document.getElementById('loginView').classList.add('hidden');
    document.getElementById('erView').classList.remove('hidden');
  }
  applyLanguage();
</script>
</body>
</html>
"""

class RelayRequestHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        return

    def _send_json(self, obj: Any, status: int = 200):
        raw = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def do_GET(self):
        path = urlparse(self.path).path
        if path in ("/", "/er"):
            raw = HTML_PAGE.encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.send_header("Content-Length", str(len(raw)))
            self.end_headers()
            self.wfile.write(raw)
        elif path == "/api/cases":
            with STORE_LOCK:
                self._send_json(list(CASES_STORE))
        else:
            self._send_json({"error": "Not found"}, 404)

    def do_POST(self):
        path = urlparse(self.path).path
        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length).decode("utf-8") or "{}")
        try:
            if path == "/api/extract":
                self._send_json(api_extract_fields(body.get("text", ""), body.get("lang", "en"), body.get("mode", "deep")))
            elif path == "/api/transcribe_audio":
                self._send_json(api_transcribe_audio(body.get("audio_b64", ""), body.get("mime_type", "audio/webm"), body.get("lang", "en")))
            elif path == "/api/draft":
                self._send_json(api_generate_draft(body))
            elif path == "/api/confirm":
                self._send_json(api_confirm_case(body))
            else:
                self._send_json({"ok": False, "error": "Unknown endpoint"}, 404)
        except Exception as e:
            self._send_json({"ok": False, "error": str(e)}, 500)

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

PORT = find_free_port()
server = ThreadingHTTPServer(("0.0.0.0", PORT), RelayRequestHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()

print(f"✅ RELAY Server running on port {PORT}")
try:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, path="/", anchor_text="🚀 Open RELAY Login & Workspace (Full Tab)")
    output.serve_kernel_port_as_window(PORT, path="/er", anchor_text="🏥 Open ER Live Dashboard (/er)")
except Exception:
    print(f"Open http://127.0.0.1:{PORT}/ and http://127.0.0.1:{PORT}/er")

✅ RELAY Server running on port 35179
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>